In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

# 数据导入

In [ ]:
order = pd.read_csv('数据1-order_detail.csv')
coupon = pd.read_csv('数据2-user_coupon_receive.csv')
visit = pd.read_csv('数据3-user_visit_detail.csv')

# 数据清洗

In [ ]:
order_cleaned = order[~(order['Coupon_id'].isna() ^ order['Coupon_type'].isna())]
coupon_cleaned = coupon[coupon.Coupon_amt != 9999]
coupon_cleaned=coupon_cleaned.dropna()
#coupon_cleaned = coupon_cleaned[coupon_cleaned['Coupon_status'] != 3]
#coupon_cleaned['Coupon_status'] = coupon_cleaned['Coupon_status'].apply(lambda x: 1 if x == 2 else 0)
visit_cleaned = visit
#coupon_cleaned=pd.read_csv('D:\管院学习\\25春\美团商分赛\选题六\df2_cleaned.csv')
#order_cleaned=pd.read_csv('D:\管院学习\\25春\美团商分赛\选题六\df1_cleaned.csv')
#visit_cleaned=pd.read_csv('D:\管院学习\\25春\美团商分赛\选题六\\06 美团业务选题-优惠券投放提效分析-数据3-user_visit_detail.csv')
date_columns = {
    'order_cleaned': ['Pay_date'],
    'coupon_cleaned': ['Receive_date', 'Start_date', 'End_date'],
    'visit_cleaned': ['Visit_date']
}
for df_name, cols in date_columns.items():
    if df_name == 'order_cleaned':
        order_cleaned[cols] = order_cleaned[cols].apply(pd.to_datetime)
    elif df_name == 'coupon_cleaned':
        coupon_cleaned[cols] = coupon_cleaned[cols].apply(pd.to_datetime)
    elif df_name == 'visit_cleaned':
        visit_cleaned[cols] = visit_cleaned[cols].apply(pd.to_datetime)

# EDA

## 券的使用情况分析

In [ ]:
import matplotlib.pyplot as plt

# 计算优惠券使用状态
coupon_usage = coupon_cleaned['Coupon_status'].value_counts().reset_index()
coupon_usage.columns = ['状态', '数量']
coupon_usage['状态'] = coupon_usage['状态'].map({1: '未使用', 2: '已使用', 3: '其他'})

# 绘制饼图
plt.figure(figsize=(8, 8))  # 调整大小以适应饼图
plt.pie(
    coupon_usage['数量'],  # 数据
    labels=coupon_usage['状态'],  # 标签
    autopct='%.1f%%',  # 显示百分比（保留1位小数）
    startangle=90,  # 起始角度（使"未使用"在顶部）
    colors=sns.color_palette('pastel'),  # 使用 seaborn 的柔和配色
    wedgeprops={'edgecolor': 'white', 'linewidth': 1}  # 边缘样式
)

plt.title('优惠券使用状态分布', fontsize=14)
plt.tight_layout()  # 防止标签溢出
plt.show()

In [ ]:
# 计算每个用户领券数量
user_coupon_counts = coupon.groupby('User_id')['Coupon_id'].count().div(6).reset_index(name='coupon_count')
# 分布直方图（对数坐标）
plt.figure(figsize=(12, 6))
sns.histplot(data=user_coupon_counts, 
             x='coupon_count', 
             bins=50,
             log_scale=True,
             color='#4e79a7')
plt.title('用户月均领券数量分布（对数坐标）', pad=20)
plt.xlabel('领券数量（对数）')
plt.ylabel('用户数')
plt.grid(axis='y', linestyle='--')
plt.show()

## 不同面额券的使用率

In [ ]:
 # 3.3 不同面额券的使用率
coupon_cleaned['coupon_amt_bin'] = pd.cut(coupon_cleaned['Coupon_amt'], 
                                           bins=[0, 5, 10, 20, 50, 100, np.inf], 
                                           labels=['0-5元', '5-10元', '10-20元', '20-50元', '50-100元', '100元以上'])
    
usage_by_amt = coupon_cleaned.groupby('coupon_amt_bin')['Coupon_status'].apply(lambda x: (x == 2).mean()).reset_index()
usage_by_amt.columns = ['优惠券面额', '使用率']
    
plt.figure(figsize=(12, 6))
sns.barplot(x='优惠券面额', y='使用率', data=usage_by_amt)
plt.title('不同面额优惠券的使用率')


# 不同有效期长度的使用率


In [ ]:
coupon_cleaned['valid_days'] = coupon_cleaned['End_date'] - coupon_cleaned['Start_date']

In [ ]:
coupon_cleaned['valid_days'] = coupon_cleaned['valid_days'].dt.days.astype(int)

In [ ]:
coupon_cleaned['valid_days_bin'] = pd.cut(coupon_cleaned['valid_days'], 
                                           bins=[0, 1, 3, 7, 14, 30, np.inf], 
                                           labels=['1天内', '1-3天', '3-7天', '7-14天', '14-30天', '30天以上'])
    
usage_by_valid = coupon_cleaned.groupby('valid_days_bin')['Coupon_status'].apply(lambda x: (x == 2).mean()).reset_index()
usage_by_valid.columns = ['有效期', '使用率']
    
plt.figure(figsize=(12, 6))
sns.barplot(x='有效期', y='使用率', data=usage_by_valid)
plt.title('不同有效期长度优惠券的使用率')

# 不同门槛券的使用率


In [ ]:
coupon_cleaned['price_limit_bin'] = pd.cut(coupon_cleaned['Price_limit'], 
                                            bins=[0, 20, 50, 100, 200, np.inf], 
                                            labels=['0-20元', '20-50元', '50-100元', '100-200元', '200元以上'])
    
usage_by_limit = coupon_cleaned.groupby('price_limit_bin')['Coupon_status'].apply(lambda x: (x == 2).mean()).reset_index()
usage_by_limit.columns = ['使用门槛', '使用率']
    
plt.figure(figsize=(12, 6))
sns.barplot(x='使用门槛', y='使用率', data=usage_by_limit)
plt.title('不同使用门槛优惠券的使用率')

# 机器学习

## 构建特征

### 特征1-4

In [ ]:
def calculate_redemption_rate(coupon_df):
    total_received = coupon_df.groupby('User_id')['Coupon_id'].nunique().reset_index(name='total_received')
    redeemed_count = coupon_df[coupon_df['Coupon_status'] == 2].groupby('User_id')['Coupon_id'].nunique().reset_index(name='redeemed_count')
    redemption_rate = pd.merge(total_received, redeemed_count, on='User_id', how='left').fillna(0)
    redemption_rate['redemption_rate'] = redemption_rate['redeemed_count'] / redemption_rate['total_received']
    return redemption_rate[['User_id', 'redemption_rate']]
feature1 = calculate_redemption_rate(coupon_cleaned)

In [ ]:
feature1

In [ ]:
def calculate_median_order_value(order_df):
    valid_orders = order_df[order_df['Actual_pay'] > 0]
    median_values = valid_orders.groupby('User_id')['Actual_pay'].median().reset_index(name='median_order_value')
    return median_values
feature2 = calculate_median_order_value(order_cleaned)

In [ ]:
feature2

In [ ]:
def calculate_visit_frequency(visit_df):
    visit_df['month'] = visit_df['Visit_date'].dt.to_period('M')
    monthly_activity = visit_df.groupby(['User_id', 'month'])['Visit_date'].nunique().reset_index(name='active_days')
    visit_freq = monthly_activity.groupby('User_id')['active_days'].mean().reset_index(name='visit_frequency')
    return visit_freq
feature3 = calculate_visit_frequency(visit_cleaned)

In [ ]:
feature3

In [ ]:
# 计算每个用户的总订单数
total_orders = order_cleaned.groupby('User_id')['Order_id'].nunique().reset_index()
total_orders.columns = ['User_id', 'total_orders']

# 计算每个用户的用券订单数（Coupon_id非空）
coupon_used = order_cleaned[order_cleaned['Coupon_id'].notna()].groupby('User_id')['Order_id'].nunique().reset_index()
coupon_used.columns = ['User_id', 'used_orders']

# 合并数据并填充未用券的用户为0
merged = pd.merge(total_orders, coupon_used, on='User_id', how='left')
merged['used_orders'] = merged['used_orders'].fillna(0)

# 计算用券频率
merged['coupon_preference'] = merged['used_orders'] / merged['total_orders']

# 生成结果（保留两位小数）
feature4 = merged[['User_id', 'coupon_preference']].round(2)
feature4.columns = ['User_id', 'coupon_preference']

In [ ]:
feature4

### 特征5-9

In [ ]:
# 执行左外连接（保留coupon_cleaned所有行）
merged_df = pd.merge(coupon_cleaned, order_cleaned, 
                    how='left', 
                    on=['User_id', 'Coupon_id'],  # 假设这两列是连接键
                    suffixes=('_df2', '_df1'))

# 显示连接结果
print(f"coupon_cleaned行数: {len(coupon_cleaned)}")
print(f"合并后行数: {len(merged_df)}")
display(merged_df.head())

In [ ]:
# 确保日期列是datetime格式
merged_df['Start_date'] = pd.to_datetime(merged_df['Start_date'])
merged_df['End_date'] = pd.to_datetime(merged_df['End_date'])

# 计算天数差并添加新列
merged_df['valid_time'] = (merged_df['End_date'] - merged_df['Start_date']).dt.days

In [ ]:
feature5_9 = merged_df[['User_id','Coupon_id','Coupon_amt','Reduce_amount','Coupon_status','Price_limit','valid_time']]

In [ ]:
feature5_9

### 特征10-13

In [ ]:
feature10 = merged_df[['User_id','Coupon_id','Receive_date','End_date']]
# 确保日期列是datetime格式
feature10['End_date'] = pd.to_datetime(feature10['End_date'])
feature10['Receive_date'] = pd.to_datetime(feature10['Receive_date'])

# 计算天数差并添加新列
feature10['Receive_to_End'] = (feature10['End_date'] - feature10['Receive_date']).dt.days
feature10=feature10[['User_id','Coupon_id','Receive_to_End']]

In [ ]:
feature10

In [ ]:
feature11 = merged_df[['User_id','Coupon_id','Receive_date']]
import cupy as cp
from numba import cuda
df3=visit_cleaned
# 确保日期列是datetime格式
feature11['Receive_date'] = pd.to_datetime(feature11['Receive_date'])
df3['Visit_date'] = pd.to_datetime(df3['Visit_date'])  # 请将date_column替换为df3中实际的日期列

cuda.select_device(0)  # 强制选择第一个NVIDIA GPU设备

with cuda.gpus[0]:  # 明确在第一个GPU上执行
    user_ids = cp.array(df3['User_id'].astype('category').cat.codes.values)
    visit_dates = cp.array(df3['Visit_date'].astype('int64') // 10**9)
    
    feature11_user_ids = cp.array(feature11['User_id'].astype('category').cat.codes.values)
    feature11_receive_dates = cp.array(feature11['Receive_date'].astype('int64') // 10**9)

# 将数据转移到GPU
d_user_ids = cuda.to_device(user_ids.get())
d_visit_dates = cuda.to_device(visit_dates.get())
seven_days_sec = 7 * 24 * 60 * 60

# 定义CUDA核函数
@cuda.jit
def count_visits_kernel(user_ids, visit_dates, query_ids, query_dates, results, seven_days_sec):
    i = cuda.grid(1)
    if i < query_ids.size:
        count = 0
        for j in range(user_ids.size):
            if (user_ids[j] == query_ids[i] and 
                visit_dates[j] >= (query_dates[i] - seven_days_sec) and 
                visit_dates[j] < query_dates[i]):
                count += 1
        results[i] = count

# 准备结果数组
results = np.zeros(len(feature11), dtype=np.int32)
d_results = cuda.to_device(results)

# 调用核函数
threads_per_block = 256
blocks_per_grid = (len(feature11) + threads_per_block - 1) // threads_per_block
with cuda.gpus[0]:
    count_visits_kernel[blocks_per_grid, threads_per_block](
        d_user_ids, d_visit_dates, 
        feature11_user_ids.get(), feature11_receive_dates.get(),
        d_results, seven_days_sec
    )

# 获取结果
feature11['prev_7days_count'] = d_results.copy_to_host()
feature11 = feature11[['User_id','Coupon_id','prev_7days_count']]

In [ ]:
feature11

In [ ]:
feature12=merged_df[['User_id','Coupon_id','Price_limit','Actual_pay','Reduce_amount']]
feature12['Actual_pay_median'] = feature12.groupby('User_id')['Actual_pay'].transform('median')
feature12['cross_1'] = feature12["Price_limit"]-feature12["Actual_pay_median"]
feature12=feature12[['User_id','Coupon_id','cross_1']]

In [ ]:
feature12

In [ ]:
feature13=merged_df[['User_id','Coupon_id','Start_date','End_date']]
# 确保日期列是datetime格式
feature13['End_date'] = pd.to_datetime(feature13['End_date'])
feature13['Start_date'] = pd.to_datetime(feature13['Start_date'])

# 计算天数差并添加新列
feature13['valid_time'] = (feature13['End_date'] - feature13['Start_date']).dt.days
# 计算每个User_id的月均活跃次数
user_counts = df3['User_id'].value_counts().rename('user_count_in_df3')

# 将统计结果合并到feature13
feature13 = feature13.merge(user_counts, left_on='User_id', right_index=True, how='left')

# 处理可能存在的NaN值(如果df3中没有该User_id)
feature13['user_count_in_df3'] = feature13['user_count_in_df3'].fillna(0).astype(int)

feature13['valid_time']=feature13['valid_time']+1
feature13['cross_2']=feature13['user_count_in_df3']/feature13['valid_time']/6
feature13=feature13[['User_id','Coupon_id','cross_2']]

In [ ]:
feature13

### 合并特征1-13

In [ ]:
features = feature5_9

In [ ]:
features = features.merge(feature1, how='left', on=['User_id'])
features = features.merge(feature2, how='left', on=['User_id'])
features = features.merge(feature3, how='left', on=['User_id'])
features = features.merge(feature4, how='left', on=['User_id'])
features = features.merge(feature10, how='left', on=['User_id','Coupon_id'])
features = features.merge(feature11, how='left', on=['User_id','Coupon_id'])
features = features.merge(feature12, how='left', on=['User_id','Coupon_id'])
features = features.merge(feature13, how='left', on=['User_id','Coupon_id'])


In [ ]:
features.isna().sum()

In [ ]:
features['median_order_value'] = features['median_order_value'].fillna(features['median_order_value'].mean())
features['visit_frequency'] = features['visit_frequency'].fillna(features['visit_frequency'].mean())
features['coupon_preference'] = features['coupon_preference'].fillna(features['coupon_preference'].mean())

# 检查特征相关性

In [ ]:
# 选择13个特征列（排除User_id和Coupon_id）
feature_cols = [col for col in features.columns if col not in ['User_id', 'Coupon_id', 'Coupon_status']]
# 处理Reduce_amount缺失值（用中位数填充）
features['Reduce_amount'] = features['Reduce_amount'].fillna(0)
# 计算相关系数矩阵
corr_matrix = features[feature_cols].corr()
# 绘制热力图
plt.figure(figsize=(15, 12))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap='coolwarm', 
            annot_kws={'size':8}, vmin=-1, vmax=1)
plt.title('13个特征的相关性热力图')
plt.xticks(rotation=45)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# 舍弃多余特征

In [ ]:
features = features.drop(columns=['cross_1','Receive_to_End'])

In [ ]:
# 选择13个特征列（排除User_id和Coupon_id）
feature_cols = [col for col in features.columns if col not in ['User_id', 'Coupon_id', 'Coupon_status']]
# 处理Reduce_amount缺失值（用中位数填充）
features['Reduce_amount'] = features['Reduce_amount'].fillna(0)
# 计算相关系数矩阵
corr_matrix = features[feature_cols].corr()
# 绘制热力图
plt.figure(figsize=(15, 12))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap='coolwarm', 
            annot_kws={'size':8}, vmin=-1, vmax=1)
plt.title('13个特征的相关性热力图')
plt.xticks(rotation=45)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# 归一化

In [ ]:
feature_scaled = features.copy()
from sklearn.preprocessing import StandardScaler
# Identify columns to scale (excluding non-feature columns)
feature_cols = [col for col in features.columns if col not in ['Coupon_status','User_id', 'Coupon_id']]
# Scale only the feature columns
scaler = StandardScaler()
feature_scaled[feature_cols] = scaler.fit_transform(features[feature_cols])
# Now feature_scaled is a DataFrame with the scaled features and original non-feature columns

In [ ]:
feature_scaled.isna().sum()

# 训练模型

## 查看features

In [ ]:
features

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import joblib
import time

# 1. 数据预处理
print("="*50)
print("数据预处理")
print("="*50)
start_time = time.time()

# 取前10000行数据加速测试（实际使用时可以去掉.head(10000)）
df = features.copy()

# 准备特征和目标变量
X = df.drop(['User_id', 'Coupon_id', 'Coupon_status'], axis=1)
y = df['Coupon_status']

# LightGBM需要类别从0开始
le = LabelEncoder()
y = le.fit_transform(y)

# 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print(f"数据预处理完成，耗时: {time.time()-start_time:.2f}秒")

# 2. 配置LightGBM GPU参数
params = {
    'objective': 'multiclass',
    'num_class': 3,
    'metric': 'multi_logloss',
    
    # GPU加速参数
    'device': 'gpu',
    
    # 模型参数
    'learning_rate': 0.05,
    'max_depth': 8,
    'num_leaves': 31,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': -1,
    'seed': 42
}

# 3. 创建数据集和训练模型
print("\n" + "="*50)
print("训练LightGBM模型")
print("="*50)

# 创建LightGBM数据集
train_data = lgb.Dataset(X_train, label=y_train)
test_data = lgb.Dataset(X_test, label=y_test, reference=train_data)

start_time = time.time()

# 训练模型（修正后的版本）
model = lgb.train(
    params,
    train_data,
    num_boost_round=500,
    valid_sets=[test_data],
    callbacks=[
        lgb.early_stopping(stopping_rounds=20),  # 早期停止
        lgb.log_evaluation(50)  # 每50轮打印一次日志
    ]
)

train_time = time.time() - start_time
print(f"模型训练完成，耗时: {train_time:.2f}秒")

# 4. 模型评估
print("\n" + "="*50)
print("模型评估")
print("="*50)

# 预测
start_time = time.time()
y_pred = model.predict(X_test, num_iteration=model.best_iteration)
y_pred_class = np.argmax(y_pred, axis=1)
predict_time = time.time() - start_time

# 评估指标
print("\n分类报告:")
print(classification_report(y_test, y_pred_class, target_names=['未使用', '已使用', '其他']))

print("\n混淆矩阵:")
print(confusion_matrix(y_test, y_pred_class))

print(f"\n准确率: {accuracy_score(y_test, y_pred_class):.4f}")
print(f"预测耗时: {predict_time:.2f}秒")

# 5. 特征重要性分析
print("\n" + "="*50)
print("特征重要性分析")
print("="*50)

# 获取特征重要性
importance = model.feature_importance(importance_type='gain')
feature_names = model.feature_name()
importance_df = pd.DataFrame({'feature': feature_names, 'importance': importance})
importance_df = importance_df.sort_values('importance', ascending=False)

# 可视化Top20特征
plt.figure(figsize=(12, 8))
plt.barh(importance_df['feature'][:20][::-1], importance_df['importance'][:20][::-1])
plt.title('LightGBM Feature Importance (Top 20)')
plt.xlabel('Importance (Gain)')
plt.tight_layout()
plt.show()

# 6. 模型保存
print("\n" + "="*50)
print("模型保存")
print("="*50)

# 保存模型
joblib.dump(model, 'lgbm_gpu_model.pkl')
joblib.dump(le, 'label_encoder.pkl')

print("模型已保存为 lgbm_gpu_model.pkl 和 label_encoder.pkl")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from imblearn.over_sampling import SMOTE
import joblib
import time
import warnings
warnings.filterwarnings('ignore')

# 1. 数据预处理
print("="*50)
print("数据预处理")
print("="*50)
start_time = time.time()

# 假设features已经定义好了
df = features.copy()  

# 准备特征和目标变量
X = df.drop(['User_id', 'Coupon_id', 'Coupon_status'], axis=1)
y = df['Coupon_status']

# LightGBM需要类别从0开始
le = LabelEncoder()
y = le.fit_transform(y)

# 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y  # 使用stratify确保训练集和测试集有相似的类别分布
)

# 查看类别分布
unique, counts = np.unique(y_train, return_counts=True)
print("训练集类别分布:", dict(zip(le.inverse_transform(unique), counts)))

# 使用SMOTE处理类别不平衡问题
print("应用SMOTE进行过采样...")
smote = SMOTE(random_state=42)
# 实际执行SMOTE过采样
try:
    X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)
    print("SMOTE过采样成功应用")
except Exception as e:
    print(f"SMOTE过采样失败: {e}")
    print("将使用原始数据集继续")
    X_train_resampled, y_train_resampled = X_train, y_train

unique, counts = np.unique(y_train_resampled, return_counts=True)
print("重采样后训练集类别分布:", dict(zip(le.inverse_transform(unique), counts)))

print(f"数据预处理完成，耗时: {time.time()-start_time:.2f}秒")

# 2. 配置LightGBM GPU参数
params = {
    'objective': 'multiclass',
    'num_class': 3,
    'metric': 'multi_logloss',
    
    # GPU加速参数
    'device': 'gpu',
    
    # 模型参数 - 优化后的参数
    'learning_rate': 0.03,
    'max_depth': 10,
    'num_leaves': 63,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'min_child_samples': 20,
    
    # 类别不平衡处理
    'is_unbalance': True,  # 处理不平衡数据
    
    'verbose': -1,
    'seed': 42
}

# 根据训练集中"已使用"类别的比例计算class_weight
class_counts = np.bincount(y_train_resampled)
total_samples = len(y_train_resampled)
class_weights = [total_samples / (len(class_counts) * count) for count in class_counts]
# 特别提高"已使用"类的权重（假设索引为1）
class_weights[1] *= 1.5

# 添加到参数中
if len(class_counts) == 3:  # 确保有3个类别
    params['class_weight'] = class_weights

# 3. 创建数据集和训练模型
print("\n" + "="*50)
print("训练LightGBM模型")
print("="*50)

# 创建LightGBM数据集
train_data = lgb.Dataset(X_train_resampled, label=y_train_resampled)
test_data = lgb.Dataset(X_test, label=y_test, reference=train_data)

start_time = time.time()

# 训练模型
model = lgb.train(
    params,
    train_data,
    num_boost_round=1000,  # 增加轮数
    valid_sets=[test_data],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50),  # 增加早停轮数
        lgb.log_evaluation(50)
    ]
)

train_time = time.time() - start_time
print(f"模型训练完成，耗时: {train_time:.2f}秒")

# 4. 模型评估
print("\n" + "="*50)
print("模型评估")
print("="*50)

# 预测
start_time = time.time()
y_pred = model.predict(X_test, num_iteration=model.best_iteration)
y_pred_class = np.argmax(y_pred, axis=1)
predict_time = time.time() - start_time

# 自定义阈值，提高"已使用"类的召回率
# 降低"已使用"(类别1)的预测阈值，使模型更容易将样本归类为"已使用"
custom_thresholds = [0.4, 0.3, 0.4]  # 为类别1降低阈值
y_pred_custom = np.zeros_like(y_pred_class)

for i in range(len(y_pred)):
    max_class = None
    max_prob = -1
    
    for j in range(3):
        if y_pred[i][j] > custom_thresholds[j] and y_pred[i][j] > max_prob:
            max_prob = y_pred[i][j]
            max_class = j
    
    # 如果没有超过阈值的类别，选择概率最高的
    if max_class is None:
        max_class = np.argmax(y_pred[i])
    
    y_pred_custom[i] = max_class

# 评估指标 - 原始预测
print("\n原始预测分类报告:")
print(classification_report(y_test, y_pred_class, target_names=['未使用', '已使用', '其他']))

print("\n原始预测混淆矩阵:")
print(confusion_matrix(y_test, y_pred_class))

print(f"\n原始预测准确率: {accuracy_score(y_test, y_pred_class):.4f}")

# 评估指标 - 自定义阈值预测
print("\n自定义阈值预测分类报告:")
print(classification_report(y_test, y_pred_custom, target_names=['未使用', '已使用', '其他']))

print("\n自定义阈值预测混淆矩阵:")
print(confusion_matrix(y_test, y_pred_custom))

print(f"\n自定义阈值预测准确率: {accuracy_score(y_test, y_pred_custom):.4f}")
print(f"预测耗时: {predict_time:.2f}秒")

# 评估"已使用"类别的F1分数改进
f1_original = f1_score(y_test, y_pred_class, average=None)[1]  # 类别1的F1分数
f1_custom = f1_score(y_test, y_pred_custom, average=None)[1]   # 类别1的F1分数
print(f"\n'已使用'类别F1分数 - 原始: {f1_original:.4f}, 自定义阈值: {f1_custom:.4f}")
print(f"'已使用'类别F1分数改进: {((f1_custom-f1_original)/f1_original)*100:.2f}%")

# 5. 特征重要性分析
print("\n" + "="*50)
print("特征重要性分析")
print("="*50)

# 获取特征重要性
importance = model.feature_importance(importance_type='gain')
feature_names = model.feature_name()
importance_df = pd.DataFrame({'feature': feature_names, 'importance': importance})
importance_df = importance_df.sort_values('importance', ascending=False)

# 可视化Top20特征
plt.figure(figsize=(12, 8))
plt.barh(importance_df['feature'][:20][::-1], importance_df['importance'][:20][::-1])
plt.title('LightGBM Feature Importance (Top 20)')
plt.xlabel('Importance (Gain)')
plt.tight_layout()
plt.savefig('feature_importance.png')  # 保存图表到文件
plt.show()

# 打印Top 10特征及其重要性
print("\nTop 10特征重要性:")
for i, (feature, imp) in enumerate(zip(importance_df['feature'][:10], importance_df['importance'][:10])):
    print(f"{i+1}. {feature}: {imp:.2f}")

# 6. 模型保存
print("\n" + "="*50)
print("模型保存")
print("="*50)

# 保存模型
joblib.dump(model, 'lgbm_gpu_model_improved.pkl')
joblib.dump(le, 'label_encoder.pkl')

print("模型已保存为 lgbm_gpu_model_improved.pkl 和 label_encoder.pkl")

# 7. 生成每个类别的ROC曲线
print("\n" + "="*50)
print("ROC曲线分析")
print("="*50)

try:
    from sklearn.metrics import roc_curve, auc
    from sklearn.preprocessing import label_binarize
    
    # 将标签转换为二进制格式
    y_test_bin = label_binarize(y_test, classes=[0, 1, 2])
    n_classes = y_test_bin.shape[1]
    
    # 计算每个类的ROC曲线和ROC面积
    fpr = dict()
    tpr = dict()
    roc_auc = dict()
    
    plt.figure(figsize=(10, 8))
    
    for i in range(n_classes):
        fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_pred[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])
        
        plt.plot(fpr[i], tpr[i], lw=2,
                 label=f'{le.inverse_transform([i])[0]} (AUC = {roc_auc[i]:.4f})')
    
    plt.plot([0, 1], [0, 1], 'k--', lw=2)
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curves for Multi-class')
    plt.legend(loc="lower right")
    plt.grid(alpha=0.3)
    plt.savefig('roc_curves.png')
    plt.show()
    
    print("各类别AUC值:")
    for i in range(n_classes):
        print(f"{le.inverse_transform([i])[0]}: {roc_auc[i]:.4f}")
        
except Exception as e:
    print(f"无法生成ROC曲线: {e}")

print("\n" + "="*50)
print("优化总结")
print("="*50)
print("1. 修复了SMOTE实现，添加了错误处理")
print("2. 使用class_weight代替scale_pos_weight处理类别不平衡")
print("3. 优化模型参数")
print("4. 自定义预测阈值提高'已使用'类的召回率")
print("5. 增加ROC曲线分析")
print("6. 添加Top 10特征重要性打印")

# 8. 预测概率分布分析
print("\n" + "="*50)
print("预测概率分布分析")
print("="*50)

try:
    # 提取每个类别的预测概率
    probs_class0 = y_pred[:, 0]  # 未使用
    probs_class1 = y_pred[:, 1]  # 已使用
    probs_class2 = y_pred[:, 2]  # 其他
    
    # 创建3个子图
    fig, axs = plt.subplots(1, 3, figsize=(18, 6))
    
    # 绘制每个类别的概率分布
    axs[0].hist(probs_class0, bins=50, alpha=0.7)
    axs[0].set_title('未使用 类别概率分布')
    axs[0].set_xlabel('预测概率')
    axs[0].set_ylabel('样本数量')
    axs[0].grid(alpha=0.3)
    
    axs[1].hist(probs_class1, bins=50, alpha=0.7)
    axs[1].set_title('已使用 类别概率分布')
    axs[1].set_xlabel('预测概率')
    axs[1].grid(alpha=0.3)
    
    axs[2].hist(probs_class2, bins=50, alpha=0.7)
    axs[2].set_title('其他 类别概率分布')
    axs[2].set_xlabel('预测概率')
    axs[2].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('probability_distributions.png')
    plt.show()
    
    # 展示每个真实类别的平均预测概率
    print("\n真实类别的平均预测概率:")
    for true_class in range(3):
        mask = (y_test == true_class)
        avg_probs = np.mean(y_pred[mask], axis=0)
        class_name = le.inverse_transform([true_class])[0]
        print(f"真实类别 '{class_name}':")
        for i, prob in enumerate(avg_probs):
            pred_class_name = le.inverse_transform([i])[0]
            print(f"  预测为'{pred_class_name}'的平均概率: {prob:.4f}")
    
except Exception as e:
    print(f"无法生成概率分布图: {e}")